[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fedgutcor/lab-ia-intermedio/blob/main/proyecto/plantilla_proyecto.ipynb)

# Plantilla de proyecto: de la pregunta a un modelo que se puede defender

**Proyecto** · Laboratorio de IA · nivel intermedio

En esta receta vas a:
- escribir el brief de tu proyecto
- cargar tus datos (o un dataset del laboratorio)
- comparar una línea base con un modelo bajo el mismo protocolo
- dejar listos el registro, el modelo guardado y un README

**Cómo usarla:** en Colab elige *Entorno de ejecución → Ejecutar todo*. No hace falta instalar nada. Si Colab avisa que el notebook no es de Google, elige *Ejecutar de todos modos*. Cuando termine, busca la celda **«Cambia esto»**, modifica un valor y vuelve a ejecutar.

> Viene llena con un ejemplo que funciona (mensajes de soporte). Cambia las celdas marcadas **TU PROYECTO** y vuelve a ejecutar todo. Esta plantilla es para **texto**; si tu proyecto es de imágenes, parte de la receta 03.

## 1 · Brief · TU PROYECTO

In [ ]:
BRIEF = {
    "usuario": "Equipo de soporte de una tienda en línea",
    "decision": "A qué cola enviar cada mensaje para que una persona lo atienda",
    "tarea": "Clasificar mensajes por motivo",
    "etiquetas": ["acceso a la cuenta", "devolución", "envío", "facturación"],
    "error_que_importa": "Mandar un problema de acceso a la cola de envíos (la persona queda bloqueada)",
    "minimo": "Un clasificador que supere a la línea base en la partición por grupo",
    "fuera_de_alcance": "Responder automáticamente al cliente",
    "datos": "Sintéticos del laboratorio (sin datos de personas)",
}

## 2 · Datos · TU PROYECTO

Tu CSV necesita una columna de texto, una de etiqueta y una de agrupación: los casos que se parecen demasiado (mismo cliente, misma frase, mismo objeto) deben quedar del mismo lado de la partición. En los datos del laboratorio, `frase` da la prueba más exigente y `grupo` una más fácil.

In [ ]:
import time, json, datetime, os
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

def como_texto(x):
    # Keras 3 recibe el texto como tensor de TensorFlow, no como columna de pandas
    return tf.constant([str(t) for t in x])

INICIO = time.time()
keras.utils.set_random_seed(0)
DATOS = "https://raw.githubusercontent.com/Fedgutcor/lab-ia-intermedio/main/datos/mensajes_soporte.csv"   # o sube tu CSV a Colab y pon su nombre
TEXTO, ETIQUETA, GRUPO = "mensaje", "motivo", "frase"

df = pd.read_csv(DATOS)
CLASES = sorted(df[ETIQUETA].unique())
df["y"] = df[ETIQUETA].map({c: i for i, c in enumerate(CLASES)})
print(len(df), "filas ·", dict(df[ETIQUETA].value_counts()))

## 3 · Partición por la columna de agrupación

In [ ]:
i_ent, i_pru = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0).split(df, groups=df[GRUPO]))
ent, pru = df.iloc[i_ent], df.iloc[i_pru]
print(f"Entrenamiento {len(ent)} · Prueba {len(pru)}")

## 4 · Línea base y dos modelos con el mismo protocolo

Ningún modelo gana siempre: compara y quédate con el que mejor atiende el error que importa.

In [ ]:
REGISTRO = "registro_experimentos.csv"
def registrar(nombre, params, exactitud):
    fila = {"fecha": datetime.datetime.now().isoformat(timespec="seconds"), "modelo": nombre,
            "parametros": json.dumps(params, ensure_ascii=False), "exactitud": round(float(exactitud), 3)}
    pd.DataFrame([fila]).to_csv(REGISTRO, mode="a", header=not os.path.exists(REGISTRO), index=False)

mayoritaria = int(ent["y"].mode()[0])
registrar("linea_base", {"clase": CLASES[mayoritaria]}, (pru["y"] == mayoritaria).mean())

def embedding(p):      # vectores de palabras promediados
    vec = layers.TextVectorization(max_tokens=3000, output_sequence_length=24); vec.adapt(ent[TEXTO].values)
    m = keras.Sequential([keras.Input((1,), dtype="string"), vec, layers.Embedding(vec.vocabulary_size(), p["dimension"]),
                          layers.GlobalAveragePooling1D(), layers.Dense(len(CLASES), activation="softmax")])
    m.compile(optimizer=keras.optimizers.Adam(p["tasa"]), loss="sparse_categorical_crossentropy"); return m

def tfidf(p):          # bolsa de palabras y pares de palabras
    vec = layers.TextVectorization(max_tokens=5000, ngrams=p["ngramas"], output_mode="tf_idf"); vec.adapt(ent[TEXTO].values)
    m = keras.Sequential([keras.Input((1,), dtype="string"), vec, layers.Dense(len(CLASES), activation="softmax")])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy"); return m

CANDIDATOS = {   # TU PROYECTO: prueba otros valores
    "embedding": (embedding, {"dimension": 16, "tasa": 0.01, "epocas": 30}),
    "tfidf": (tfidf, {"ngramas": 2, "epocas": 20}),
}
modelos = {}
for nombre, (crear, p) in CANDIDATOS.items():
    m = crear(p)
    m.fit(como_texto(ent[TEXTO]), ent["y"].values, epochs=p["epocas"], batch_size=32, verbose=0)
    pred = m.predict(como_texto(pru[TEXTO]), verbose=0).argmax(axis=1)
    registrar(nombre, p, (pru["y"].values == pred).mean())
    modelos[nombre] = (m, pred)
    print(f"── {nombre}\n" + classification_report(pru["y"], pred, target_names=CLASES, digits=3, zero_division=0))
pd.read_csv(REGISTRO)

## 5 · Elegir un modelo

In [ ]:
exactitudes = {n: float((pru["y"].values == pred).mean()) for n, (m, pred) in modelos.items()}
ELEGIDO = max(exactitudes, key=exactitudes.get)    # TU PROYECTO: elige por el error que importa, no solo por el promedio
modelo = modelos[ELEGIDO][0]
print("Exactitud por modelo:", {n: round(v, 3) for n, v in exactitudes.items()}, "→ elegido:", ELEGIDO)

## 6 · Guardar y contrato de uso

In [ ]:
modelo.save("mi_modelo.keras")
VERSION = f"mi-modelo-{ELEGIDO}-" + datetime.date.today().isoformat()

def predecir(texto):
    if not isinstance(texto, str) or not texto.strip():
        return {"error": "La entrada debe ser un texto no vacío."}
    p = modelo.predict(como_texto([texto]), verbose=0)[0]
    return {"etiqueta": CLASES[int(p.argmax())], "confianza": round(float(p.max()), 3), "version": VERSION}

print(predecir("No puedo entrar a mi cuenta desde ayer"))
print(predecir(""))

## 7 · README del proyecto (se genera solo; complétalo con tus límites)

In [ ]:
registro = pd.read_csv(REGISTRO)
tabla = "| modelo | exactitud | parámetros |\n|---|---|---|\n" + "\n".join(
    f"| {r.modelo} | {r.exactitud} | {r.parametros} |" for r in registro.itertuples())
readme = f"""# {BRIEF['tarea']}

**Usuario:** {BRIEF['usuario']}
**Decisión que apoya:** {BRIEF['decision']}
**Etiquetas:** {', '.join(BRIEF['etiquetas'])}
**Error que más importa:** {BRIEF['error_que_importa']}
**Fuera de alcance:** {BRIEF['fuera_de_alcance']}
**Datos:** {BRIEF['datos']} · partición por `{GRUPO}`

## Resultados (misma prueba para todos)

{tabla}

## Cómo usarlo

`predecir(texto)` devuelve etiqueta, confianza y versión ({VERSION}). Rechaza textos vacíos.

## Límites

- TU PROYECTO: escribe qué NO puede afirmarse con esta prueba (tamaño, origen de los datos, casos no cubiertos).
"""
open("README_proyecto.md", "w", encoding="utf-8").write(readme)
print(readme)
print(f"Tiempo total: {time.time() - INICIO:.0f} s")